# Bank Marketing Subscription Prediction
## Customer targeting with machine learning

**Business problem:** A bank wants to prioritize customers who are more likely to subscribe to a term deposit, so campaign resources can be focused more effectively.

**Project workflow**
- Audit the dataset and investigate target imbalance and categorical values.
- Build leakage-aware preprocessing pipelines for Logistic Regression, Random Forest, and XGBoost.
- Evaluate models using precision, recall, F1-score, ROC-AUC, and confusion matrices.
- Choose a probability threshold using validation data, then evaluate once on the untouched test set.
- Translate predictions into targeting segments and illustrative campaign economics.
- Interpret the final XGBoost model with feature importance and SHAP.

**Important modeling choices**
- `duration` is excluded because call duration is only known after a call and would not be available when prioritizing customers before contact.
- `campaign` is excluded for this pre-contact prioritization scenario because it represents contacts during the current campaign.
- The target is imbalanced, so accuracy alone is not sufficient. Threshold selection should reflect the trade-off between contacting more potential subscribers and avoiding unproductive contacts.

**Reproducibility:** Run the notebook from top to bottom in Google Colab. The dataset is downloaded from the UCI Machine Learning Repository.


> **Evaluation protocol:** the dataset is split into 60% training, 20% validation, and 20% final test data. Early model comparisons and threshold exploration use the validation set. The final test set is held aside until the final model and threshold are selected. Because the data are historical and ordered by campaign time, a chronological/out-of-time validation should also be tested before any real deployment.

## 1. Download and load the dataset

In [ ]:
from pathlib import Path
import urllib.request
import zipfile
import pandas as pd

# Download the official UCI archive.
url = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
data_dir = Path("/content/bank_marketing")
archive_path = Path("/content/bank_marketing.zip")
data_dir.mkdir(parents=True, exist_ok=True)

urllib.request.urlretrieve(url, archive_path)
with zipfile.ZipFile(archive_path, "r") as archive:
    archive.extractall(data_dir)

# The UCI archive contains nested ZIP files. Extract them recursively so this
# works even if the archive's folder layout changes slightly.
while True:
    nested_archives = list(data_dir.rglob("*.zip"))
    if not nested_archives:
        break
    for nested_archive in nested_archives:
        with zipfile.ZipFile(nested_archive, "r") as archive:
            archive.extractall(nested_archive.parent)
        nested_archive.unlink()

matches = list(data_dir.rglob("bank-additional-full.csv"))
if not matches:
    raise FileNotFoundError(
        "Could not find bank-additional-full.csv inside the downloaded UCI archive."
    )

file_path = matches[0]
bank_raw = pd.read_csv(file_path, sep=";")
if "y" not in bank_raw.columns:
    raise ValueError("The downloaded dataset does not contain the expected target column 'y'.")

print("Dataset loaded from:", file_path)
print("Dataset shape:", bank_raw.shape)
print("Target column found:", "y" in bank_raw.columns)
bank_raw.head()


## 2. Initial data audit and target distribution

In [ ]:
print("Dataset shape:", bank_raw.shape)
print("\nColumn names:")
print(bank_raw.columns.tolist())

In [ ]:
# Display column names, data types, and non-null counts
bank_raw.info()

In [ ]:
# Count missing values in each column
missing_values = bank_raw.isnull().sum()

print(missing_values[missing_values > 0])

In [ ]:
# Count the number of clients in each target category
print(bank_raw["y"].value_counts())

In [ ]:
# Calculate the percentage of clients in each target category
print(
    (bank_raw["y"].value_counts(normalize=True) * 100).round(2)
)

In [ ]:
import matplotlib.pyplot as plt

target_counts = bank_raw["y"].value_counts().reindex(["no", "yes"])

target_counts.plot(kind="bar", figsize=(7, 4))

plt.title("Term Deposit Subscription Distribution")
plt.xlabel("Subscription Outcome")
plt.ylabel("Number of Clients")
plt.xticks(rotation=0)

plt.show()

In [ ]:
# Display descriptive statistics for numeric columns
bank_raw.describe().T

In [ ]:
# Select columns containing categorical data
categorical_cols = bank_raw.select_dtypes(include=["object"]).columns

# Count "unknown" values in each categorical column
unknown_counts = (
    bank_raw[categorical_cols]
    .eq("unknown")
    .sum()
    .sort_values(ascending=False)
)

# Display only columns containing "unknown"
print("Unknown values in categorical columns:")
print(unknown_counts[unknown_counts > 0])

In [ ]:
# Display the number of unique categories in each categorical column
for col in categorical_cols:
    print(f"\n{col}: {bank_raw[col].nunique()} unique categories")
    print(bank_raw[col].value_counts().head(10))

## 3. Prepare features and split the data

In [ ]:
# Separate input features from the target variable

X = bank_raw.drop(
    columns=["y", "duration", "campaign"]
)

# Convert the target into numerical labels
y = bank_raw["y"].map({
    "no": 0,
    "yes": 1
})

# Inspect the results
print("Input features shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nFirst 5 rows of input features:")
display(X.head())

In [ ]:
from sklearn.model_selection import train_test_split

# First reserve 20% as the final test set.
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Split the remaining 80% into 60% training and 20% validation.
X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.25,
    random_state=42,
    stratify=y_dev
)

print("Training features:", X_train.shape)
print("Validation features:", X_val.shape)
print("Final test features:", X_test.shape)
print("Training target:", y_train.shape)
print("Validation target:", y_val.shape)
print("Final test target:", y_test.shape)

print("\nTarget rates:")
print("Train:", round(y_train.mean(), 4))
print("Validation:", round(y_val.mean(), 4))
print("Test:", round(y_test.mean(), 4))

## 4. Preprocessing pipeline

In [ ]:
# Identify numerical columns
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

# Identify categorical columns
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:", numeric_features)
print("\nNumber of numerical features:", len(numeric_features))

print("\nCategorical features:", categorical_features)
print("\nNumber of categorical features:", len(categorical_features))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# Preprocessing for numerical features
numeric_transformer = StandardScaler()

# Preprocessing for categorical features
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

# Combine both preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully!")

## 5. Baseline model — Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Create a complete machine learning pipeline
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train the model
model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

In [ ]:
# Predict customer subscription outcomes
y_pred = model.predict(X_val)

# Predict subscription probabilities
y_prob = model.predict_proba(X_val)[:, 1]

print("First 10 actual outcomes:")
print(y_val.iloc[:10].values)

print("\nFirst 10 predicted outcomes:")
print(y_pred[:10])

print("\nFirst 10 predicted subscription probabilities:")
print(y_prob[:10].round(3))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Calculate evaluation metrics
accuracy = accuracy_score(y_val, y_pred)

precision = precision_score(
    y_val, y_pred, zero_division=0
)

recall = recall_score(
    y_val, y_pred, zero_division=0
)

f1 = f1_score(
    y_val, y_pred, zero_division=0
)

roc_auc = roc_auc_score(y_val, y_prob)

# Display results
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Calculate the confusion matrix
cm = confusion_matrix(y_val, y_pred)

print("Confusion Matrix:")
print(cm)

# Visualise the results
display_matrix = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Subscription", "Subscription"]
)

display_matrix.plot()
plt.title("Logistic Regression - Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

threshold = 0.3

y_pred_03 = (y_prob >= threshold).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_03))

print("\nAccuracy:", round(accuracy_score(y_val, y_pred_03), 4))
print("Precision:", round(precision_score(y_val, y_pred_03), 4))
print("Recall:", round(recall_score(y_val, y_pred_03), 4))
print("F1-score:", round(f1_score(y_val, y_pred_03), 4))

In [ ]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

thresholds = [0.2, 0.3, 0.4, 0.5]

results = []

for threshold in thresholds:
    y_pred_threshold = (y_prob >= threshold).astype(int)

    results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_val, y_pred_threshold),
        "Precision": precision_score(y_val, y_pred_threshold),
        "Recall": recall_score(y_val, y_pred_threshold),
        "F1-score": f1_score(y_val, y_pred_threshold)
    })

threshold_results = pd.DataFrame(results)

threshold_results.round(4)

## 6. Initial threshold exploration and campaign framing

> **Illustrative only:** the contact cost (₹50) and value per subscription (₹2,000) are placeholders. Historical subscribers are not necessarily incremental outcomes caused by a campaign contact, so these calculations are not verified profit or causal uplift.


In [ ]:
threshold_summary = []

for threshold in [0.2, 0.3, 0.4, 0.5]:
    y_pred_threshold = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val, y_pred_threshold
    ).ravel()

    threshold_summary.append({
        "Threshold": threshold,
        "Customers Contacted": int(tp + fp),
        "True Subscribers Identified": int(tp),
        "False Positives": int(fp),
        "Subscribers Missed": int(fn)
    })

threshold_summary_df = pd.DataFrame(threshold_summary)

threshold_summary_df

In [ ]:
contact_cost = 50
subscription_value = 2000

threshold_summary_df["Campaign Cost"] = (
    threshold_summary_df["Customers Contacted"] * contact_cost
)

threshold_summary_df["Subscription Value"] = (
    threshold_summary_df["True Subscribers Identified"] * subscription_value
)

threshold_summary_df["Net Campaign Value"] = (
    threshold_summary_df["Subscription Value"]
    - threshold_summary_df["Campaign Cost"]
)

threshold_summary_df["ROI (%)"] = (
    threshold_summary_df["Net Campaign Value"]
    / threshold_summary_df["Campaign Cost"]
) * 100

threshold_summary_df.round(2)

In [ ]:
customer_scores = X_val.copy()

customer_scores["Actual_Outcome"] = y_val.values
customer_scores["Predicted_Probability"] = y_prob

customer_scores = customer_scores.sort_values(
    by="Predicted_Probability",
    ascending=False
).reset_index(drop=True)

customer_scores[
    ["Actual_Outcome", "Predicted_Probability"]
].head(10)

In [ ]:
budget_customers = 500

top_customers = customer_scores.head(budget_customers)

total_contacted = len(top_customers)

actual_subscribers = top_customers["Actual_Outcome"].sum()

observed_conversion_rate = actual_subscribers / total_contacted

print("Customers contacted:", total_contacted)
print("Actual subscribers:", actual_subscribers)
print(
    "Observed conversion rate:",
    round(observed_conversion_rate * 100, 2),
    "%"
)

In [ ]:
# Calculate the overall subscription rate in the validation set
baseline_conversion_rate = y_val.mean()

# Expected subscribers if we selected 500 customers randomly
expected_random_subscribers = (
    baseline_conversion_rate * len(top_customers)
)

# Compare the model with the baseline
print(
    "validation-set subscription rate:",
    round(baseline_conversion_rate * 100, 2),
    "%"
)

print(
    "Expected subscribers from random targeting:",
    round(expected_random_subscribers, 1)
)

print(
    "Subscribers identified by model:",
    int(top_customers["Actual_Outcome"].sum())
)

print(
    "Lift over random targeting:",
    round(
        observed_conversion_rate / baseline_conversion_rate,
        2
    ),
    "x"
)

## 7. Ranking customers and cumulative gains

In [ ]:
import numpy as np
import pandas as pd

# Rank customers from highest to lowest predicted probability
gains_df = customer_scores.sort_values(
    by="Predicted_Probability",
    ascending=False
).reset_index(drop=True)

# Calculate cumulative number of actual subscribers
gains_df["Cumulative_Subscribers"] = (
    gains_df["Actual_Outcome"].cumsum()
)

# Calculate cumulative percentage of customers selected
gains_df["Cumulative_Customers_Pct"] = (
    np.arange(1, len(gains_df) + 1) / len(gains_df) * 100
)

# Calculate cumulative percentage of all actual subscribers captured
total_subscribers = gains_df["Actual_Outcome"].sum()

gains_df["Cumulative_Gains_Pct"] = (
    gains_df["Cumulative_Subscribers"]
    / total_subscribers
    * 100
)

gains_df.head(10)

In [ ]:
# Correct cumulative gains to percentage values
gains_df["Cumulative_Gains_Pct"] = (
    gains_df["Cumulative_Subscribers"]
    / total_subscribers
    * 100
)

gains_df[
    [
        "Cumulative_Subscribers",
        "Cumulative_Customers_Pct",
        "Cumulative_Gains_Pct"
    ]
].head(10).round(3)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.plot(
    gains_df["Cumulative_Customers_Pct"],
    gains_df["Cumulative_Gains_Pct"],
    label="Logistic Regression (ranking model)"
)

plt.plot(
    [0, 100],
    [0, 100],
    linestyle="--",
    label="Random targeting"
)

plt.xlabel("Percentage of customers targeted")
plt.ylabel("Percentage of actual subscribers captured")
plt.title("Cumulative Gains Chart — Logistic Regression")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Evaluate cumulative gains at different targeting levels

targeting_levels = [10, 20, 30, 50]

gains_summary = []

for pct in targeting_levels:
    n_customers = int(len(gains_df) * pct / 100)

    selected = gains_df.head(n_customers)

    subscribers_captured = int(
        selected["Actual_Outcome"].sum()
    )

    gains_pct = (
        subscribers_captured / total_subscribers
    ) * 100

    gains_summary.append({
        "Customers_Targeted_Pct": pct,
        "Customers_Selected": n_customers,
        "Subscribers_Captured": subscribers_captured,
        "Cumulative_Gains_Pct": round(gains_pct, 2)
    })

gains_summary_df = pd.DataFrame(gains_summary)

gains_summary_df

## 8. Random Forest comparison

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully!")

In [ ]:
# Predict customer outcomes
y_pred_rf = rf_model.predict(X_val)

# Predict subscription probabilities
y_prob_rf = rf_model.predict_proba(X_val)[:, 1]

print("First 10 actual outcomes:", y_val.values[:10])
print("First 10 predictions:", y_pred_rf[:10])
print("First 10 probabilities:", y_prob_rf[:10].round(3))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("Accuracy:", round(accuracy_score(y_val, y_pred_rf), 4))
print("Precision:", round(precision_score(y_val, y_pred_rf), 4))
print("Recall:", round(recall_score(y_val, y_pred_rf), 4))
print("F1-score:", round(f1_score(y_val, y_pred_rf), 4))
print("ROC-AUC:", round(roc_auc_score(y_val, y_prob_rf), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_rf))

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.2, 0.3, 0.4, 0.5]

for threshold in thresholds:
    y_pred_threshold = (y_prob_rf >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")
    print("Customers selected:", y_pred_threshold.sum())
    print("Precision:", round(precision_score(y_val, y_pred_threshold), 4))
    print("Recall:", round(recall_score(y_val, y_pred_threshold), 4))
    print("F1-score:", round(f1_score(y_val, y_pred_threshold), 4))

In [ ]:
from sklearn.metrics import f1_score

# Compare thresholds consistently using validation F1-score.
threshold_candidates = [0.2, 0.3, 0.4, 0.5]


def best_validation_threshold(y_true, y_probability, candidates):
    scores = []
    for candidate in candidates:
        predictions = (y_probability >= candidate).astype(int)
        scores.append((f1_score(y_true, predictions, zero_division=0), candidate))
    # If F1 is tied, prefer the higher threshold to contact fewer customers.
    return sorted(scores, key=lambda item: (item[0], item[1]), reverse=True)[0][1]


## 9. XGBoost comparison

In [ ]:
%pip install xgboost

In [ ]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_model.fit(X_train, y_train)

print("XGBoost model trained successfully!")

In [ ]:
# Predict customer outcomes: 0 or 1
y_pred_xgb = xgb_model.predict(X_val)

# Predict subscription probabilities
y_prob_xgb = xgb_model.predict_proba(X_val)[:, 1]

print("First 10 actual outcomes:", y_val.values[:10])
print("First 10 predictions:", y_pred_xgb[:10])
print("First 10 probabilities:", y_prob_xgb[:10].round(3))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("Accuracy:", round(accuracy_score(y_val, y_pred_xgb), 4))
print("Precision:", round(precision_score(y_val, y_pred_xgb), 4))
print("Recall:", round(recall_score(y_val, y_pred_xgb), 4))
print("F1-score:", round(f1_score(y_val, y_pred_xgb), 4))
print("ROC-AUC:", round(roc_auc_score(y_val, y_prob_xgb), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_xgb))

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.2, 0.3, 0.4, 0.5]

for threshold in thresholds:
    y_pred_threshold = (y_prob_xgb >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")
    print("Customers selected:", y_pred_threshold.sum())
    print("Precision:", round(precision_score(y_val, y_pred_threshold), 4))
    print("Recall:", round(recall_score(y_val, y_pred_threshold), 4))
    print("F1-score:", round(f1_score(y_val, y_pred_threshold), 4))

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

threshold_xgb = 0.2
y_pred_xgb_best = (y_prob_xgb >= threshold_xgb).astype(int)

print("XGBoost Accuracy:", round(
    accuracy_score(y_val, y_pred_xgb_best), 4
))

print("\nConfusion Matrix at threshold 0.2:")
print(confusion_matrix(y_val, y_pred_xgb_best))

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score

# Training predictions
y_train_pred_xgb = xgb_model.predict(X_train)
y_train_prob_xgb = xgb_model.predict_proba(X_train)[:, 1]

# Validation predictions (already calculated)
y_val_pred_xgb = xgb_model.predict(X_val)

print("Training F1-score:",
      round(f1_score(y_train, y_train_pred_xgb), 4))

print("Validation F1-score:",
      round(f1_score(y_val, y_val_pred_xgb), 4))

print("Training ROC-AUC:",
      round(roc_auc_score(y_train, y_train_prob_xgb), 4))

print("Validation ROC-AUC:",
      round(roc_auc_score(y_val, y_prob_xgb), 4))

### Validation comparison across all three models

The table selects each model's threshold from the same predefined candidates using validation F1-score. ROC-AUC is included to compare how well each model ranks likely subscribers independently of a particular threshold. The final test set remains untouched here.

In [ ]:
# Compare all candidate models on validation data using the same threshold-selection rule.
threshold_candidates = [0.2, 0.3, 0.4, 0.5]


def validation_metrics(y_true, probabilities):
    threshold = best_validation_threshold(
        y_true, probabilities, threshold_candidates
    )
    predictions = (probabilities >= threshold).astype(int)
    return {
        "Selected threshold": threshold,
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1-score": f1_score(y_true, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, probabilities),
    }

validation_model_comparison = pd.DataFrame.from_dict(
    {
        "Logistic Regression": validation_metrics(y_val, y_prob),
        "Random Forest": validation_metrics(y_val, y_prob_rf),
        "XGBoost": validation_metrics(y_val, y_prob_xgb),
    },
    orient="index",
)
validation_model_comparison.index.name = "Model"
display(validation_model_comparison.round(4))


In [ ]:
# Confirm the three-way split created above.
print("Training samples:", X_train.shape[0])
print("Validation samples:", X_val.shape[0])
print("Final test samples:", X_test.shape[0])
print("Total samples:", len(X_train) + len(X_val) + len(X_test))

## 10. Validation-based threshold selection and final test evaluation

In [ ]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

xgb_model_final = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_model_final.fit(X_train, y_train)

print("Final XGBoost model trained on the training split!")

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# Use validation data for model selection and threshold tuning.
y_val_prob_xgb = xgb_model_final.predict_proba(X_val)[:, 1]
print("Validation ROC-AUC:", round(roc_auc_score(y_val, y_val_prob_xgb), 4))

threshold_rows = []
for threshold in [0.2, 0.3, 0.4, 0.5]:
    y_val_pred = (y_val_prob_xgb >= threshold).astype(int)
    threshold_rows.append({
        "Threshold": threshold,
        "Precision": precision_score(y_val, y_val_pred, zero_division=0),
        "Recall": recall_score(y_val, y_val_pred, zero_division=0),
        "F1-score": f1_score(y_val, y_val_pred, zero_division=0),
        "Customers selected": int(y_val_pred.sum())
    })

validation_threshold_results = pd.DataFrame(threshold_rows)
display(validation_threshold_results.round(4))

# Use the same F1-based selection and tie-break rule as the comparison table.
selected_threshold = best_validation_threshold(
    y_val, y_val_prob_xgb, [0.2, 0.3, 0.4, 0.5]
)
selected_row = validation_threshold_results.loc[
    validation_threshold_results["Threshold"] == selected_threshold
].iloc[0]
print("Selected threshold (highest validation F1):", selected_threshold)
print("Validation F1-score:", round(float(selected_row["F1-score"]), 4))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Final test evaluation: use the preselected validation threshold once.
y_test_prob_final = xgb_model_final.predict_proba(X_test)[:, 1]

# Apply the threshold selected using validation data
y_test_pred_final = (
    y_test_prob_final >= selected_threshold
).astype(int)

print("Final Held-Out Test Results")
print("Threshold:", selected_threshold)
print("Accuracy:", round(accuracy_score(y_test, y_test_pred_final), 4))
print("Precision:", round(precision_score(y_test, y_test_pred_final), 4))
print("Recall:", round(recall_score(y_test, y_test_pred_final), 4))
print("F1-score:", round(f1_score(y_test, y_test_pred_final), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_test_prob_final), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_final))

In [ ]:
# Illustrative business assumptions
contact_cost = 50
subscription_value = 2000

# Actual outcomes from our final test predictions
true_positives = (
    (y_test_pred_final == 1) & (y_test == 1)
).sum()

false_positives = (
    (y_test_pred_final == 1) & (y_test == 0)
).sum()

total_contacted = (y_test_pred_final == 1).sum()

# Calculate campaign economics
campaign_cost = total_contacted * contact_cost
subscription_value_total = true_positives * subscription_value
net_value = subscription_value_total - campaign_cost

print("Customers contacted:", total_contacted)
print("Successful subscriptions:", true_positives)
print("Unsuccessful contacts:", false_positives)
print("Total contact cost: ₹", campaign_cost)
print("Illustrative value attributed to subscriptions: ₹", subscription_value_total)
print("Net campaign value: ₹", net_value)

## 11. Illustrative campaign economics and targeting lift

In [ ]:
import numpy as np

# Define the target group using the final model's actual threshold-based predictions.
selected_mask = np.asarray(y_test_pred_final == 1)
budget_customers = int(selected_mask.sum())
model_subscribers = int(y_test.to_numpy()[selected_mask].sum())

# Expected subscribers if the same number of customers were contacted at random.
baseline_conversion_rate = float(y_test.mean())
expected_random_subscribers = budget_customers * baseline_conversion_rate

print("Customers selected by the model:", budget_customers)
print("Subscribers among selected customers:", model_subscribers)
print("Baseline conversion rate:", round(baseline_conversion_rate, 4))
print("Expected subscribers with random targeting:",
      round(expected_random_subscribers, 2))


In [ ]:
# Calculate targeting improvement

model_conversion_rate = model_subscribers / budget_customers

lift = model_conversion_rate / baseline_conversion_rate

additional_subscribers = (
    model_subscribers - expected_random_subscribers
)

print("Model conversion rate:", round(model_conversion_rate * 100, 2), "%")
print("Lift over random targeting:", round(lift, 2), "x")
print("Additional subscribers vs random:",
      round(additional_subscribers, 2))

In [ ]:
# Business comparison: model targeting vs random targeting

contact_cost = 50
subscription_value = 2000

# Model-based targeting
model_contacts = budget_customers
model_subscribers_count = model_subscribers

model_campaign_cost = model_contacts * contact_cost
model_subscription_value = model_subscribers_count * subscription_value
model_net_value = model_subscription_value - model_campaign_cost

# Random-targeting expectation
random_contacts = budget_customers
random_subscribers_count = expected_random_subscribers

random_campaign_cost = random_contacts * contact_cost
random_subscription_value = random_subscribers_count * subscription_value
random_net_value = random_subscription_value - random_campaign_cost

print("MODEL-BASED TARGETING")
print("Campaign cost: ₹", round(model_campaign_cost))
print("Subscription value: ₹", round(model_subscription_value))
print("Modelled net value: ₹", round(model_net_value))

print("\nRANDOM TARGETING")
print("Campaign cost: ₹", round(random_campaign_cost))
print("Expected subscription value: ₹", round(random_subscription_value))
print("Expected net value: ₹", round(random_net_value))

print("\nDifference in modelled net value: ₹",
      round(model_net_value - random_net_value))

## 12. XGBoost feature importance

In [ ]:
import pandas as pd

# Get the trained XGBoost classifier from the pipeline
xgb_classifier = xgb_model_final.named_steps["classifier"]

# Get feature names after preprocessing
feature_names = (
    xgb_model_final.named_steps["preprocessor"]
    .get_feature_names_out()
)

# Create a feature importance table
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": xgb_classifier.feature_importances_
})

# Display the 15 most important features
importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

importance_df.head(15)

In [ ]:
!pip install shap

In [ ]:
import shap

# Get the preprocessor and trained XGBoost classifier
preprocessor = xgb_model_final.named_steps["preprocessor"]
xgb_classifier = xgb_model_final.named_steps["classifier"]

# Transform the test data into the format used by XGBoost
X_test_transformed = preprocessor.transform(X_test)

# Get the feature names after encoding
feature_names = preprocessor.get_feature_names_out()

# Create a SHAP explainer
explainer = shap.TreeExplainer(xgb_classifier)

# Calculate SHAP values
shap_values = explainer(X_test_transformed)

print("SHAP values calculated successfully!")
print("Shape:", shap_values.values.shape)

## 13. SHAP model interpretation

In [ ]:
import pandas as pd
import shap
import matplotlib.pyplot as plt

# Convert transformed test data into a DataFrame
X_test_shap = pd.DataFrame(
    X_test_transformed.toarray()
    if hasattr(X_test_transformed, "toarray")
    else X_test_transformed,
    columns=feature_names,
    index=X_test.index
)

# Plot SHAP values with feature values
shap.summary_plot(
    shap_values.values,
    X_test_shap,
    max_display=15
)

In [ ]:
# Examine previous-contact history

print("Customers never previously contacted (pdays = 999):")
print((bank_raw["pdays"] == 999).sum())

print("\nCustomers previously contacted (pdays != 999):")
print((bank_raw["pdays"] != 999).sum())

print("\nPercentage never previously contacted:")
print(round((bank_raw["pdays"] == 999).mean() * 100, 2), "%")

## 14. Exploratory analysis: previous campaign history

In [ ]:
# Create groups based on previous-contact history

bank_raw["previous_contact_status"] = (
    bank_raw["pdays"]
    .apply(lambda x: "Never contacted" if x == 999
           else "Previously contacted")
)

# Calculate customer counts and subscription rates
contact_analysis = (
    bank_raw.groupby("previous_contact_status")
    .agg(
        total_customers=("y", "count"),
        subscribers=("y", lambda x: (x == "yes").sum()),
        subscription_rate=("y", lambda x: (x == "yes").mean() * 100)
    )
    .round(2)
)

contact_analysis

In [ ]:
poutcome_analysis = (
    bank_raw.groupby("poutcome")
    .agg(
        total_customers=("y", "count"),
        subscribers=("y", lambda x: (x == "yes").sum()),
        subscription_rate=("y", lambda x: (x == "yes").mean() * 100)
    )
    .round(2)
    .sort_values("subscription_rate", ascending=False)
)

poutcome_analysis

In [ ]:
import numpy as np

# Evaluate threshold-based predictions by job category on the held-out test set.
group_test = X_test.copy()
group_test["actual"] = y_test.to_numpy()
group_test["predicted"] = y_test_pred_final

group_test["true_positives"] = (
    (group_test["actual"] == 1) & (group_test["predicted"] == 1)
).astype(int)

group_test["predicted_positives"] = group_test["predicted"].astype(int)

job_performance = group_test.groupby("job").agg(
    customers=("actual", "size"),
    actual_subscribers=("actual", "sum"),
    true_positives=("true_positives", "sum"),
    predicted_positives=("predicted_positives", "sum"),
)

job_performance["recall"] = (
    job_performance["true_positives"]
    / job_performance["actual_subscribers"].replace(0, np.nan)
).fillna(0)
job_performance["precision"] = (
    job_performance["true_positives"]
    / job_performance["predicted_positives"].replace(0, np.nan)
).fillna(0)

job_performance = job_performance.drop(
    columns=["true_positives", "predicted_positives"]
).sort_values("recall", ascending=False)

job_performance.round(3)


## 15. Segment-level model performance

In [ ]:
from sklearn.metrics import confusion_matrix

# Calculate the confusion matrix
cm = confusion_matrix(y_test, y_test_pred_final)

tn, fp, fn, tp = cm.ravel()

print("True Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)

print("\nTotal customers:", len(y_test))
print("Customers correctly identified:", tn + tp)
print("Incorrect predictions:", fp + fn)

## 16. Customer priority segments

In [ ]:
# Create customer priority segments

targeting_df = X_test.copy()

targeting_df["actual_subscription"] = y_test.values
targeting_df["predicted_probability"] = y_test_prob_final

targeting_df["priority"] = pd.cut(
    targeting_df["predicted_probability"],
    bins=[-0.001, 0.2, 0.5, 1.0],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

# Analyze each segment
priority_analysis = (
    targeting_df.groupby("priority", observed=True)
    .agg(
        customers=("actual_subscription", "count"),
        subscribers=("actual_subscription", "sum"),
        avg_probability=("predicted_probability", "mean")
    )
)

priority_analysis["conversion_rate"] = (
    priority_analysis["subscribers"]
    / priority_analysis["customers"] * 100
)

priority_analysis.round(3)

## 17. Limitations and recommendations

- **No causal claim:** the campaign economics compare model-selected customers with an expected random-targeting baseline. This is not a randomized experiment and does not establish incremental profit.
- **Illustrative assumptions:** the contact cost and subscription value are placeholders. Replace them with actual campaign costs and contribution margin or customer lifetime value before using the estimates for a real decision.
- **Threshold trade-off:** the selected threshold is chosen from the validation results using F1-score; a lower threshold generally captures more potential subscribers but also creates more false positives. The best threshold depends on contact capacity and the relative costs of missed subscribers versus unsuccessful contacts.
- **Historical data:** the model learns patterns in this dataset and may not generalize to a different period, bank, or campaign without validation.
- **Segment metrics:** subgroup results can be unstable for small groups and should be checked for sample size, calibration, and fairness before operational use.
- **Deployment next steps:** test the targeting policy in a controlled campaign, monitor conversion and model drift, and retrain only after checking performance on newer data.

## Conclusion

This project demonstrates an end-to-end classification workflow for bank campaign prioritization: data audit, leakage-aware preprocessing, model comparison, validation-based threshold selection, held-out test evaluation, interpretation, and business framing. The final model should be treated as a decision-support prototype until its performance and incremental business impact are validated in a live or randomized campaign.
